<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day5_7_(260602)_%EA%B0%84%EB%8B%A8_%ED%9A%8C%EC%9B%90%EA%B0%80%EC%9E%85_%EB%93%B1%EB%A1%9D_%EB%B0%8F_%EC%A1%B0%ED%9A%8C_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 간단 회원가입

In [2]:
%%writefile /content/spring-lab/simple-signup/build.gradle

plugins {
    id 'java'
    id 'org.springframework.boot' version '3.3.5'
    id 'io.spring.dependency-management' version '1.1.6'
}

group = 'com.example'
version = '0.0.1-SNAPSHOT'

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(17)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-thymeleaf'
    implementation 'org.springframework.boot:spring-boot-starter-jdbc'

    runtimeOnly 'org.mariadb.jdbc:mariadb-java-client'

    testImplementation 'org.springframework.boot:spring-boot-starter-test'
}

tasks.named('test') {
    useJUnitPlatform()
}

Overwriting /content/spring-lab/simple-signup/build.gradle


In [3]:
%%writefile /content/spring-lab/simple-signup/src/main/resources/application.properties

server.port=3100

spring.datasource.url=jdbc:mariadb://localhost:3306/signup_lab
spring.datasource.username=testuser
spring.datasource.password=1234
spring.datasource.driver-class-name=org.mariadb.jdbc.Driver

spring.thymeleaf.cache=false

Overwriting /content/spring-lab/simple-signup/src/main/resources/application.properties


In [4]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/dto/SignupForm.java

package com.example.demo.dto;

public class SignupForm {

    private String email;
    private String password;
    private String name;

    public SignupForm() {
    }

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }

    public String getName() {
        return name;
    }

    public void setEmail(String email) {
        this.email = email;
    }

    public void setPassword(String password) {
        this.password = password;
    }

    public void setName(String name) {
        this.name = name;
    }
}

Writing /content/spring-lab/simple-signup/src/main/java/com/example/demo/dto/SignupForm.java


In [5]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/domain/Member.java

package com.example.demo.domain;

import java.time.LocalDateTime;

public class Member {

    private Long id;
    private String email;
    private String password;
    private String name;
    private LocalDateTime createdAt;

    public Member(
            Long id,
            String email,
            String password,
            String name,
            LocalDateTime createdAt
    ) {
        this.id = id;
        this.email = email;
        this.password = password;
        this.name = name;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public String getEmail() {
        return email;
    }

    public String getPassword() {
        return password;
    }

    public String getName() {
        return name;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}


Writing /content/spring-lab/simple-signup/src/main/java/com/example/demo/domain/Member.java


In [6]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/repository/MemberRepository.java

package com.example.demo.repository;

import com.example.demo.domain.Member;
import com.example.demo.dto.SignupForm;
import org.springframework.dao.DuplicateKeyException;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.jdbc.core.RowMapper;
import org.springframework.stereotype.Repository;

import java.util.List;

@Repository
public class MemberRepository {

    private final JdbcTemplate jdbcTemplate;

    public MemberRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    private final RowMapper<Member> memberRowMapper = (rs, rowNum) -> new Member(
            rs.getLong("id"),
            rs.getString("email"),
            rs.getString("password"),
            rs.getString("name"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    public void save(SignupForm form) {
        String sql = """
                INSERT INTO members (email, password, name)
                VALUES (?, ?, ?)
                """;

        jdbcTemplate.update(
                sql,
                form.getEmail(),
                form.getPassword(),
                form.getName()
        );
    }

    public List<Member> findAll() {
        String sql = """
                SELECT id, email, password, name, created_at
                FROM members
                ORDER BY id DESC
                """;

        return jdbcTemplate.query(sql, memberRowMapper);
    }
}


Writing /content/spring-lab/simple-signup/src/main/java/com/example/demo/repository/MemberRepository.java


In [7]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/service/SignupService.java

package com.example.demo.service;

import com.example.demo.domain.Member;
import com.example.demo.dto.SignupForm;
import com.example.demo.repository.MemberRepository;
import org.springframework.dao.DuplicateKeyException;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class CustomerService {

    private final MemberRepository repository;

    public CustomerService(MemberRepository repository) {
        this.repository = repository;
    }

    public void signup(SignupForm form) {
        validate(form);

        try {
            repository.save(form);
        } catch (DuplicateKeyException e) {
            throw new IllegalArgumentException("이미 가입된 이메일입니다.");
        }
    }

    public List<Member> findMembers() {
        return repository.findAll();
    }

    private void validate(SignupForm form) {
        if (form.getEmail() == null || form.getEmail().isBlank()) {
            throw new IllegalArgumentException("이메일을 입력해 주세요.");
        }

        if (form.getPassword() == null || form.getPassword().isBlank()) {
            throw new IllegalArgumentException("비밀번호를 입력해 주세요.");
        }

        if (form.getName() == null || form.getName().isBlank()) {
            throw new IllegalArgumentException("이름을 입력해 주세요.");
        }
    }
}

Writing /content/spring-lab/simple-signup/src/main/java/com/example/demo/service/SignupService.java


In [42]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/controller/SignupController.java

package com.example.demo.controller;

import com.example.demo.dto.SignupForm;
import com.example.demo.service.SignupService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;

@Controller
public class SignupController {

    private final SignupService service;

    public SignupController(SignupService service) {
        this.service = service;
    }

    //@GetMapping("/")
    //public String home() {
        //return "redirect:/members";
    //}

    @GetMapping("/members")
    public String members(Model model) {
        model.addAttribute("signupForm", new SignupForm());
        model.addAttribute("members", service.findMembers());

        return "members";
    }

    @PostMapping("/members")
    public String signup(@ModelAttribute SignupForm form, Model model) {
        try {
            service.signup(form);
            return "redirect:/members";
        } catch (IllegalArgumentException e) {
            model.addAttribute("signupForm", form);
            model.addAttribute("members", service.findMembers());
            model.addAttribute("errorMessage", e.getMessage());

            return "members";
        }
    }
}


Overwriting /content/spring-lab/simple-signup/src/main/java/com/example/demo/controller/SignupController.java


In [9]:
%%writefile /content/spring-lab/simple-signup/src/main/resources/templates/members.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>간단 회원가입</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="page">
    <section class="card form-card">
        <h1>간단 회원가입</h1>
        <p class="description">
            이메일, 비밀번호, 이름을 입력하면 DTO, Controller, Service, Repository를 거쳐 MariaDB에 저장됩니다.
        </p>

        <div class="error-box" th:if="${errorMessage != null}" th:text="${errorMessage}">
            오류 메시지
        </div>

        <form action="/members" method="post" th:object="${signupForm}">
            <div class="field">
                <label>이메일</label>
                <input type="email" th:field="*{email}" placeholder="user@test.com">
            </div>

            <div class="field">
                <label>비밀번호</label>
                <input type="password" th:field="*{password}" placeholder="비밀번호">
            </div>

            <div class="field">
                <label>이름</label>
                <input type="text" th:field="*{name}" placeholder="홍길동">
            </div>

            <button type="submit">회원가입</button>
        </form>
    </section>

    <section class="card list-card">
        <h2>회원 목록</h2>
        <p class="description">
            DB에서 조회된 데이터가 Member Domain 객체로 변환된 뒤 화면에 출력됩니다.
        </p>

        <table>
            <thead>
            <tr>
                <th>ID</th>
                <th>이메일</th>
                <th>비밀번호</th>
                <th>이름</th>
                <th>가입시각</th>
            </tr>
            </thead>
            <tbody>
            <tr th:each="member : ${members}">
                <td th:text="${member.id}">1</td>
                <td th:text="${member.email}">user@test.com</td>
                <td th:text="${member.password}">1234</td>
                <td th:text="${member.name}">홍길동</td>
                <td th:text="${#temporals.format(member.createdAt, 'yyyy-MM-dd HH:mm:ss')}">
                    2026-01-01 10:00:00
                </td>
            </tr>
            </tbody>
        </table>
    </section>
</div>
</body>
</html>


Writing /content/spring-lab/simple-signup/src/main/resources/templates/members.html


# 간단 등록·조회 구조 응용 실습 주제

In [43]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/dto/CustomerForm.java

package com.example.demo.dto;

public class CustomerForm {

    private String email;
    private String phone;
    private String name;

    public CustomerForm() {
    }

    public String getEmail() {
        return email;
    }

    public String getPhone() {
        return phone;
    }

    public String getName() {
        return name;
    }

    public void setEmail(String email) {
        this.email = email;
    }

    public void setPhone(String phone) {
        this.phone = phone;
    }

    public void setName(String name) {
        this.name = name;
    }
}


Overwriting /content/spring-lab/simple-signup/src/main/java/com/example/demo/dto/CustomerForm.java


In [38]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/domain/Customer.java

package com.example.demo.domain;

import java.time.LocalDateTime;

public class Customer {

    private Long id;
    private String email;
    private String phone;
    private String name;
    private LocalDateTime createdAt;

    public Customer(
            Long id,
            String name,
            String email,
            String phone,
            LocalDateTime createdAt
    ) {
        this.id = id;
        this.name = name;
        this.email = email;
        this.phone = phone;
        this.createdAt = createdAt;
    }

    public Long getId() {
        return id;
    }

    public String getEmail() {
        return email;
    }

    public String getPhone() {
        return phone;
    }

    public String getName() {
        return name;
    }

    public LocalDateTime getCreatedAt() {
        return createdAt;
    }
}

Overwriting /content/spring-lab/simple-signup/src/main/java/com/example/demo/domain/Customer.java


In [39]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/repository/CustomerRepository.java

package com.example.demo.repository;

import com.example.demo.domain.Customer;
import com.example.demo.dto.CustomerForm;
import org.springframework.jdbc.core.JdbcTemplate;
import org.springframework.jdbc.core.RowMapper;
import org.springframework.stereotype.Repository;

import java.util.List;

@Repository
public class CustomerRepository {

    private final JdbcTemplate jdbcTemplate;

    public CustomerRepository(JdbcTemplate jdbcTemplate) {
        this.jdbcTemplate = jdbcTemplate;
    }

    private final RowMapper<Customer> customerRowMapper = (rs, rowNum) -> new Customer(
            rs.getLong("id"),
            rs.getString("customer_name"),
            rs.getString("email"),
            rs.getString("phone"),
            rs.getTimestamp("created_at").toLocalDateTime()
    );

    public void save(CustomerForm form) {
        String sql = """
                INSERT INTO customers (customer_name, email, phone)
                VALUES (?, ?, ?)
                """;

        jdbcTemplate.update(
                sql,
                form.getName(),
                form.getEmail(),
                form.getPhone()
        );
    }

    public List<Customer> findAll() {
        String sql = """
                SELECT id, customer_name, email, phone, created_at
                FROM customers
                ORDER BY id DESC
                """;

        return jdbcTemplate.query(sql, customerRowMapper);
    }
}


Overwriting /content/spring-lab/simple-signup/src/main/java/com/example/demo/repository/CustomerRepository.java


In [40]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/service/CustomerService.java

package com.example.demo.service;

import com.example.demo.domain.Customer;
import com.example.demo.dto.CustomerForm;
import com.example.demo.repository.CustomerRepository;
import org.springframework.dao.DuplicateKeyException;
import org.springframework.stereotype.Service;

import java.util.List;

@Service
public class CustomerService {

    private final CustomerRepository repository;

    public CustomerService(CustomerRepository repository) {
        this.repository = repository;
    }

    public void signup(CustomerForm form) {
        validate(form);

        try {
            repository.save(form);

        } catch (DuplicateKeyException e) {
            throw new IllegalArgumentException("이미 가입된 이메일입니다.");
        }
    }

    public List<Customer> findCustomers() {
        return repository.findAll();
    }

    private void validate(CustomerForm form) {

        if (form.getEmail() == null || form.getEmail().isBlank()) {
            throw new IllegalArgumentException("이메일을 입력해 주세요.");
        }

        if (form.getPhone() == null || form.getPhone().isBlank()) {
            throw new IllegalArgumentException("번호를 입력해 주세요.");
        }

        if (form.getName() == null || form.getName().isBlank()) {
            throw new IllegalArgumentException("이름을 입력해 주세요.");
        }
    }
}

Overwriting /content/spring-lab/simple-signup/src/main/java/com/example/demo/service/CustomerService.java


In [33]:
%%writefile /content/spring-lab/simple-signup/src/main/java/com/example/demo/controller/CustomerController.java

package com.example.demo.controller;

import com.example.demo.dto.CustomerForm;
import com.example.demo.service.CustomerService;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;

@Controller
public class CustomerController {

    private final CustomerService service;

    public CustomerController(CustomerService service) {
        this.service = service;
    }

    @GetMapping("/")
    public String home() {
        return "redirect:/customers";
    }

    @GetMapping("/customers")
    public String customers(Model model) {
        model.addAttribute("customerForm", new CustomerForm());
        model.addAttribute("customers", service.findCustomers());

        return "customers";
    }

    @PostMapping("/customers")
    public String signup(@ModelAttribute CustomerForm form, Model model) {
        try {
            service.signup(form);
            return "redirect:/customers";
        } catch (IllegalArgumentException e) {
            model.addAttribute("customerForm", form);
            model.addAttribute("customers", service.findCustomers());
            model.addAttribute("errorMessage", e.getMessage());

            return "customers";
        }
    }
}


Overwriting /content/spring-lab/simple-signup/src/main/java/com/example/demo/controller/CustomerController.java


In [41]:
%%writefile /content/spring-lab/simple-signup/src/main/resources/templates/customers.html

<!DOCTYPE html>
<html lang="ko" xmlns:th="http://www.thymeleaf.org">
<head>
    <meta charset="UTF-8">
    <title>간단 회원가입</title>
    <link rel="stylesheet" href="/style.css">
</head>
<body>
<div class="page">
    <section class="card form-card">
        <h1>간단 회원가입</h1>
        <p class="description">
            이메일, 번호, 이름을 입력하면 DTO, Controller, Service, Repository를 거쳐 MariaDB에 저장됩니다.
        </p>

        <div class="error-box" th:if="${errorMessage != null}" th:text="${errorMessage}">
            오류 메시지
        </div>

        <form action="/customers" method="post" th:object="${customerForm}">
            <div class="field">
                <label>이메일</label>
                <input type="email" th:field="*{email}" placeholder="user@test.com">
            </div>

            <div class="field">
                <label>번호</label>
                <input type="text" th:field="*{phone}" placeholder="번호">
            </div>

            <div class="field">
                <label>이름</label>
                <input type="text" th:field="*{name}" placeholder="홍길동">
            </div>

            <button type="submit">회원가입</button>
        </form>
    </section>

    <section class="card list-card">
        <h2>회원 목록</h2>
        <p class="description">
            DB에서 조회된 데이터가 Customer Domain 객체로 변환된 뒤 화면에 출력됩니다.
        </p>

        <table>
            <thead>
            <tr>
                <th>ID</th>
                <th>이메일</th>
                <th>번호</th>
                <th>이름</th>
                <th>가입시각</th>
            </tr>
            </thead>
            <tbody>
            <tr th:each="customer : ${customers}">
                <td th:text="${customer.id}">1</td>
                <td th:text="${customer.email}">user@test.com</td>
                <td th:text="${customer.phone}">010-1234-5678</td>
                <td th:text="${customer.name}">홍길동</td>
                <td th:text="${#temporals.format(customer.createdAt, 'yyyy-MM-dd HH:mm:ss')}">
                    2026-01-01 10:00:00
                </td>
            </tr>
            </tbody>
        </table>
    </section>
</div>
</body>
</html>


Overwriting /content/spring-lab/simple-signup/src/main/resources/templates/customers.html
